In [ ]:
import os
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from dotenv import load_dotenv

load_dotenv()

# 1. 初始化LLM
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY")
)


# 稳定性很高
# 1. 给他一份完型填空的结构----Pydantic数据模型----JSON SCHEMA
# 2. json 字符串很准确
# 3. 反序列化成数据模型的时候很准确性（报错）---- JSON 字符串 ---- Pydantic数据模型
# 2. 定义Pydantic模型
class CalendarEvent(BaseModel):
    name: str
    date: str
    participants: list[str]


# 3. 使用with_structured_output，返回一个新的Runnable
structured_llm = llm.with_structured_output(schema=CalendarEvent)

# 4. 调用，直接返回Pydantic对象
result = structured_llm.invoke("Alice and Bob are going to a science fair on Friday.")
print(result)  # CalendarEvent(name='Science Fair', date='Friday', participants=['Alice', 'Bob'])
print(type(result))  # <class 'CalendarEvent'>

In [ ]:
# 1.导入相关依赖
import os, json
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel, Field
from dotenv import load_dotenv

load_dotenv()
# 2. 实例化模型
llm = ChatOpenAI(
    model="gpt-4o",
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY"),
)

# 3. 实例化输出解析器
str_out_put_parser = StrOutputParser()

# 4. 调用模型(json格式字符串)
chains = llm | str_out_put_parser

chains.invoke("简单介绍一下你自己")


In [10]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field, field_validator
from langchain_openai import ChatOpenAI

class MovieReview(BaseModel):
    """电影评论结构"""
    title: str = Field(description="电影标题")
    rating: int = Field(description="评分，1-10分", ge=1, le=10)
    summary: str = Field(description="剧情简介")
    recommended: bool = Field(description="是否推荐")



llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
parser = PydanticOutputParser(pydantic_object=MovieReview)

# 将格式说明注入到Prompt中
prompt = ChatPromptTemplate.from_messages([
    ("system", parser.get_format_instructions()),
    ("human", "评价电影《盗梦空间》")
])

chain = prompt | llm | parser
result = chain.invoke({})
# print(f"电影: {result.title}, 评分: {result.rating}/10")

ValueError: Invalid format specifier in f-string template. Nested replacement fields are not allowed.